# SkyGuard AI — Iteration 11 Data Rebuild

**Run this corrected notebook, not the older `SkyGuard_AI_GPU_Iteration_11_Colab.ipynb`.**
The old ZIP named “545 stations” contained 24-station observations plus a larger station catalog.
This version actually downloads new Indian-station observations and builds fresh training tables.

Verified discovery on 11 September 2026: **545 Indian ISD station IDs; 543 have coordinates;
441 have at least one listed 2020–2023 file (1,596 station-year files).** These are availability
counts, not quality-passing counts. Discovery rechecks official metadata in a new experiment.
IMD's 1008 AWS network is a different network, not a percentage denominator for this archive.

Data: official NOAA/NCEI historical Indian surface/airport station observations, **not direct IMD AWS telemetry**.
Temperature and pressure are reported; RH here is derived from T/dew point. Dew point never enters model features.
Only T/P/RH, time, and station metadata for causal context are used. No rain/wind/forecast predictors.

This is a research baseline rebuild, not a completed production/maintenance system or guaranteed accuracy increase.
Original project datasets, old results and deployed models remain untouched.

## 1. Run instructions — pehle yeh padhein

1. Upload **only this notebook** to Google Colab. No old data ZIP is needed.
2. Run setup and connect your Google Drive. Data is downloaded automatically from official NOAA URLs.
3. Default `RUN_MODE='all_available'` processes the full discovered Indian candidate network.
   A first run may require multiple sessions: downloads and station features resume from checked files.
   Runtime is not promised to be 6–8 minutes. Keep enough free Drive space (several GB; actual size depends on files).
4. CPU runtime is sufficient for data download/features and LightGBM. For optional CatBoost select **T4 GPU**.
5. Use **Runtime → Run all**. If Colab disconnects, reconnect, run setup, then Run all again.
   Completed files are reused only when their checksums/contracts match.
6. Send back the small result ZIP from the last cell. Do not send model files unless requested.

`pilot` mode downloads six new stations for troubleshooting and deliberately does not claim national training.
At least 50 eligible stations, including 25 outside the original24, are required for the expanded training path.
This is a pragmatic readiness guard, not an SIH-prescribed threshold.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=1.26,<3', 'pandas>=2.2,<3', 'pyarrow>=16,<24',
    'scikit-learn>=1.5,<1.9', 'lightgbm==4.6.0', 'catboost>=1.2.7,<1.3',
    'joblib>=1.4,<2', 'requests>=2.31,<3'])
print('Dependencies installed. If Colab explicitly requests a restart, restart once and rerun setup.')

In [ ]:
from pathlib import Path
import json, os, sys, shutil, gc, importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import drive
drive.mount('/content/drive')

RUN_MODE = 'all_available'  # 'pilot' for six NEW stations; it does not run national training
RUN_TRAINING = True
USE_CATBOOST = True
RUN_EXTRA_SEEDS = False  # True runs two additional complete scenarios after the first run
SEED = 111
EXPERIMENT_NAME = 'iteration11_data_rebuild_v1'
I11_ROOT = Path('/content/drive/MyDrive/SkyGuard_AI_GPU/experiments') / EXPERIMENT_NAME
I11_ROOT.mkdir(parents=True, exist_ok=True)
assert RUN_MODE in {'all_available', 'pilot'}
print('Persistent experiment:', I11_ROOT)
print('Drive available GB (Colab mount may not reflect account quota):', round(shutil.disk_usage(I11_ROOT).free/1024**3, 1))
GPU_AVAILABLE = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
print('GPU available:', GPU_AVAILABLE, '| CatBoost enabled:', USE_CATBOOST)
print('LightGBM and data preparation use CPU. Unused GPU memory during these steps is normal.')

## 2. Self-contained, checksummed implementation

The next cell installs the two embedded Python modules in the temporary Colab runtime.
All experiment data stays on Drive. There are no hidden `train`, `dev`, or previous-iteration variables.
Changing embedded source changes cache contracts; use a new experiment name after substantive changes.

In [ ]:
MODULE_SOURCES = {'iteration11_data': '"""Iteration 11: auditable all-India historical station discovery and preparation.\n\nIndependent experiment: never reads labelled 2024/2025 tests or writes old models.\nNOAA ISD observations are a public Indian-station PROXY, not an IMD AWS export.\nOnly the explicit 2020--2023 historical ISD CSV schema is supported here. GHCNh\nrequires a separate parser; an HTML/error/new-format response fails validation.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport re\nimport shutil\nimport time\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport requests\n\nVERSION = "iteration11-v1"\nYEARS = (2020, 2021, 2022, 2023)\nBASE = "https://www.ncei.noaa.gov/data/global-hourly/access"\nHISTORY_URL = "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv"\nINVENTORY_URL = "https://www.ncei.noaa.gov/pub/data/noaa/isd-inventory.csv"\nPRIMARY = ["temperature_c", "pressure_hpa", "relative_humidity_pct"]\nGOOD_QC = {"0", "1", "4", "5"}  # usable QC-screened proxy, not verified healthy\nPRESSURES = ["station_pressure", "sea_level_pressure", "altimeter_pressure"]\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef atomic_replace(source, target):\n    # Cloud-synced folders can briefly lock a just-closed file on Windows.\n    for attempt in range(8):\n        try:\n            Path(source).replace(target)\n            return\n        except PermissionError:\n            if attempt == 7:\n                raise\n            time.sleep(.3 * (attempt + 1))\n\n\ndef write_json(path, value):\n    def clean(item):\n        if isinstance(item, dict):\n            return {str(k): clean(v) for k, v in item.items()}\n        if isinstance(item, (list, tuple)):\n            return [clean(v) for v in item]\n        if isinstance(item, (float, np.floating)) and not np.isfinite(item):\n            return None\n        if isinstance(item, np.generic):\n            return item.item()\n        return item\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".partial")\n    tmp.write_text(json.dumps(clean(value), indent=2, default=str, allow_nan=False), encoding="utf-8")\n    atomic_replace(tmp, path)\n\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef download(url, path, min_free_gb=2):\n    """Resumable at FILE boundaries, atomic completion, retries, content receipts."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    receipt = path.with_name(path.name + ".receipt.json")\n    if path.exists() and receipt.exists():\n        previous = json.loads(receipt.read_text())\n        if previous.get("url") == url and previous.get("sha256") == digest(path):\n            return previous\n        raise RuntimeError(f"Cached file/receipt mismatch: {path}; use a new experiment folder")\n    if path.exists():\n        raise RuntimeError(f"Unreceipted file exists: {path}; will not silently adopt/overwrite it")\n    last_error = None\n    for attempt in range(3):\n        try:\n            if shutil.disk_usage(path.parent).free < min_free_gb * 1024**3:\n                raise OSError("Insufficient disk space; pause and free space before resuming")\n            with requests.get(url, stream=True, timeout=(20, 60), headers={\n                "User-Agent": "SkyGuard-academic-data-audit/11 (bounded historical download)"\n            }) as response:\n                response.raise_for_status()\n                tmp = path.with_name(path.name + ".partial")\n                with tmp.open("wb") as stream:\n                    for block in response.iter_content(1024 * 1024):\n                        if block:\n                            stream.write(block)\n                if not tmp.stat().st_size:\n                    raise ValueError("Empty response")\n                if path.suffix == ".csv":\n                    header = pd.read_csv(tmp, nrows=0).columns\n                    required = {"STATION", "DATE", "TMP", "DEW", "SLP"} if "/access/" in url else {"USAF", "WBAN"}\n                    if not required.issubset(header):\n                        raise ValueError(f"Unsupported source schema: {list(header)[:10]}")\n                atomic_replace(tmp, path)\n                record = {"url": url, "retrieved_at_utc": utc_now(), "bytes": path.stat().st_size,\n                          "sha256": digest(path), "last_modified": response.headers.get("Last-Modified")}\n                write_json(receipt, record)\n                return record\n        except (requests.RequestException, ValueError, OSError) as exc:\n            last_error = exc\n            if isinstance(exc, OSError) and "disk space" in str(exc):\n                break\n            if attempt < 2:\n                time.sleep(2 ** attempt)\n    raise RuntimeError(f"Download failed ({url}): {last_error}")\n\n\ndef discover(root, benchmark_ids=()):\n    root = Path(root)\n    download(HISTORY_URL, root / "metadata/isd-history.csv")\n    download(INVENTORY_URL, root / "metadata/isd-inventory.csv")\n    history = pd.read_csv(root / "metadata/isd-history.csv", dtype=str).fillna("")\n    h = history.loc[history.CTRY.eq("IN")].copy()\n    h["station_id"] = h.USAF.str.zfill(6) + h.WBAN.str.zfill(5)\n    if h.station_id.duplicated().any():\n        raise ValueError("Duplicate station IDs in history: resolve before selecting stations")\n    for source, target in [("LAT", "latitude"), ("LON", "longitude"), ("ELEV(M)", "elevation_m")]:\n        h[target] = pd.to_numeric(h[source], errors="coerce")\n    h["coordinate_valid"] = h.latitude.between(-90, 90) & h.longitude.between(-180, 180) & ~((h.latitude == 0) & (h.longitude == 0))\n    h["station_name"] = h["STATION NAME"]\n    h["legacy_24_station"] = h.station_id.isin(set(benchmark_ids))\n    # Co-located station aliases must not enter opposite train/holdout sides.\n    h["geographic_block"] = (np.floor(h.latitude / 2).astype("Int64").astype(str) + ":" +\n                              np.floor(h.longitude / 2).astype("Int64").astype(str))\n    h["station_role"] = h.geographic_block.map(\n        lambda x: "spatial_holdout" if int(hashlib.sha256(("i11:" + x).encode()).hexdigest()[:8], 16) % 5 == 0 else "development")\n    inventory = pd.read_csv(root / "metadata/isd-inventory.csv", dtype={"USAF": str, "WBAN": str})\n    inventory["station_id"] = inventory.USAF.str.zfill(6) + inventory.WBAN.str.zfill(5)\n    months = ["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"]\n    jobs = []\n    for year in YEARS:\n        index_path = root / f"metadata/index_{year}.html"\n        download(f"{BASE}/{year}/", index_path)\n        listed = set(re.findall(r\'href="(\\d{11})\\.csv"\', index_path.read_text(errors="replace")))\n        if not listed:\n            raise ValueError(f"No ISD CSV links found for {year}; source availability must be reviewed")\n        counts = inventory.loc[inventory.YEAR.eq(year)].set_index("station_id")[months].sum(axis=1)\n        h[f"inventory_reports_{year}"] = h.station_id.map(counts).fillna(0).astype(int)\n        h[f"file_listed_{year}"] = h.station_id.isin(listed)\n        for sid in h.loc[h.coordinate_valid & h[f"file_listed_{year}"], "station_id"]:\n            jobs.append({"station_id": sid, "year": year, "url": f"{BASE}/{year}/{sid}.csv"})\n    h.to_csv(root / "station_catalog.csv", index=False)\n    jobs = pd.DataFrame(jobs)\n    jobs.to_csv(root / "download_plan.csv", index=False)\n    summary = {\n        "version": VERSION, "created_at_utc": utc_now(), "source": "NOAA ISD Indian-station historical proxy; not IMD AWS network",\n        "indian_metadata_station_ids": len(h), "with_coordinates": int(h.coordinate_valid.sum()),\n        "candidate_station_ids_with_any_listed_file": int(jobs.station_id.nunique()),\n        "listed_station_year_files": len(jobs), "listed_by_year": jobs.groupby("year").station_id.nunique().to_dict(),\n        "usable_T_P_RH_station_count": "NOT KNOWN until raw downloads and quality profiling complete",\n        "imd_1008": "Separate IMD network count, NOT denominator for NOAA archive coverage",\n        "no_2024_or_2025_data": True,\n    }\n    write_json(root / "discovery_receipt.json", summary)\n    return h, jobs, summary\n\n\ndef acquire(root, station_ids=None, workers=3):\n    root = Path(root)\n    jobs = pd.read_csv(root / "download_plan.csv", dtype={"station_id": str})\n    if station_ids is not None:\n        jobs = jobs[jobs.station_id.isin(station_ids)]\n    results = []\n    def fetch(row):\n        try:\n            rec = download(row["url"], root / f"raw/{row[\'year\']}/{row[\'station_id\']}.csv")\n            return {**row, **rec, "status": "complete", "error": ""}\n        except Exception as exc:\n            return {**row, "status": "failed", "error": str(exc)}\n    with ThreadPoolExecutor(max_workers=max(1, min(workers, 4))) as pool:\n        futures = [pool.submit(fetch, row) for row in jobs.to_dict("records")]\n        for done in as_completed(futures):\n            result = done.result()\n            results.append(result)\n            pd.DataFrame(results).to_csv(root / "download_status.csv", index=False)\n            if len(results) % 20 == 0 or result["status"] == "failed":\n                print(f"Downloaded/checked {len(results)}/{len(jobs)}: {result[\'station_id\']} {result[\'status\']}", flush=True)\n    return pd.DataFrame(results)\n\n\ndef parse_group(series, index=0):\n    parts = series.fillna("").astype(str).str.split(",", expand=True)\n    raw = parts[index].fillna("").str.strip() if index in parts else pd.Series("", index=series.index)\n    values = pd.to_numeric(raw, errors="coerce").div(10)\n    values = values.mask(raw.str.match(r"^[+-]?999", na=False))\n    quality = parts[index + 1].fillna("").str.strip() if index + 1 in parts else pd.Series("", index=series.index)\n    return values, quality\n\n\ndef normalize(raw, meta):\n    sid = str(meta["station_id"])\n    r = raw.loc[raw.STATION.astype(str).eq(sid)].copy()\n    o = pd.DataFrame(index=r.index)\n    o["timestamp_utc"] = pd.to_datetime(r.DATE, utc=True, errors="coerce")\n    o["temperature_c"], o["temperature_qc"] = parse_group(r.TMP)\n    dew, dew_qc = parse_group(r.DEW)\n    o["dew_point_c"] = dew  # provenance only: never a model feature\n    o["dew_point_qc"] = dew_qc\n    # No silent RH clipping: supersaturation and invalid derivations stay auditable.\n    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):\n        rh = 100 * np.exp(17.625 * dew / (243.04 + dew) - 17.625 * o.temperature_c / (243.04 + o.temperature_c))\n    o["relative_humidity_pct"] = rh.replace([np.inf, -np.inf], np.nan)\n    o["rh_source"] = "derived_from_T_and_dewpoint_NOT_direct_sensor_RH"\n    o["sea_level_pressure"], o["sea_level_pressure_qc"] = parse_group(r.SLP)\n    ma = r.get("MA1", pd.Series("", index=r.index))\n    o["altimeter_pressure"], o["altimeter_pressure_qc"] = parse_group(ma, 0)\n    o["station_pressure"], o["station_pressure_qc"] = parse_group(ma, 2)\n    for key in ["station_id", "latitude", "longitude", "elevation_m", "station_role", "geographic_block", "legacy_24_station"]:\n        o[key] = meta[key]\n    o["report_type"] = r.get("REPORT_TYPE", "")\n    o["source"] = "NOAA_NCEI_ISD"\n    o["source_qc_temperature_ok"] = o.temperature_qc.isin(GOOD_QC)\n    o["source_qc_humidity_ok"] = o.temperature_qc.isin(GOOD_QC) & dew_qc.isin(GOOD_QC)\n    o = o.loc[o.timestamp_utc.notna() & o.timestamp_utc.dt.year.isin(YEARS)]\n    # Deterministic duplicate report selection; no label information involved.\n    o["_complete"] = o[["temperature_c", "relative_humidity_pct", *PRESSURES]].notna().sum(axis=1)\n    o["_qc"] = o.source_qc_temperature_ok.astype(int) + o.source_qc_humidity_ok.astype(int)\n    o = o.sort_values(["timestamp_utc", "_qc", "_complete"], ascending=[True, False, False], kind="stable")\n    return o.drop_duplicates("timestamp_utc").drop(columns=["_complete", "_qc"]).sort_values("timestamp_utc").reset_index(drop=True)\n\n\ndef choose_pressure(frame):\n    """Choose ONE pressure type using 2020--21 only; never row-wise datum fallback."""\n    fit = frame.loc[frame.timestamp_utc.dt.year.isin([2020, 2021])]\n    counts = {p: int((fit[p].notna() & fit[p + "_qc"].isin(GOOD_QC)).sum()) for p in PRESSURES}\n    choice = max(PRESSURES, key=lambda p: counts[p]) if max(counts.values(), default=0) else None\n    o = frame.copy()\n    o["pressure_datum"] = choice or "unavailable"\n    o["pressure_hpa"] = o[choice] if choice else np.nan\n    o["source_qc_pressure_ok"] = o[choice + "_qc"].isin(GOOD_QC) if choice else False\n    plausible = o.temperature_c.between(-90, 65) & o.pressure_hpa.between(300, 1100) & o.relative_humidity_pct.between(0, 100, inclusive="right")\n    o["qc_screened_proxy"] = plausible & o.source_qc_temperature_ok & o.source_qc_humidity_ok & o.source_qc_pressure_ok\n    o["label_status"] = np.where(o.qc_screened_proxy, "presumed_normal_NOT_verified", "unknown")\n    o["ground_truth_fault"] = -1  # archived observations are NOT sensor-fault ground truth\n    return o, counts\n\n\ndef profile(frame, split):\n    d = frame.loc[frame.timestamp_utc.dt.year.isin([2020, 2021] if split == "fit" else [2022, 2023])]\n    delta = d.timestamp_utc.diff().dt.total_seconds().div(60)\n    complete = d[PRIMARY].notna().all(axis=1)\n    screened = d.qc_screened_proxy\n    return {\n        "rows": len(d), "complete_triples": int(complete.sum()), "screened_proxy_rows": int(screened.sum()),\n        "screened_days": int(d.loc[screened, "timestamp_utc"].dt.floor("D").nunique()),\n        "median_cadence_minutes": float(delta.median()) if delta.notna().any() else None,\n        "gaps_over_6h": int(delta.gt(360).sum()),\n        "missing_pressure_fraction": float(d.pressure_hpa.isna().mean()) if len(d) else None,\n    }\n\n\ndef haversine(lat1, lon1, lat2, lon2):\n    a, b, c, d = map(np.radians, [lat1, lon1, lat2, lon2])\n    x = np.sin((c-a)/2)**2 + np.cos(a)*np.cos(c)*np.sin((d-b)/2)**2\n    return 6371 * 2 * np.arcsin(np.sqrt(np.clip(x, 0, 1)))\n\n\ndef neighbor_graph(stations, radius_km=150, elevation_tolerance_m=300, max_neighbors=8):\n    edges = []\n    rows = stations.to_dict("records")\n    for a in rows:\n        options = []\n        for b in rows:\n            if a["station_id"] == b["station_id"]:\n                continue\n            if not np.isfinite(a["elevation_m"]) or not np.isfinite(b["elevation_m"]):\n                continue  # no invented zero elevation\n            km = float(haversine(a["latitude"], a["longitude"], b["latitude"], b["longitude"]))\n            dz = abs(a["elevation_m"] - b["elevation_m"])\n            if km <= radius_km and dz <= elevation_tolerance_m:\n                # Exclude near-coincident aliases; they are not independent buddies.\n                if km < 1:\n                    continue\n                options.append({"station_id": a["station_id"], "neighbor_id": b["station_id"],\n                                "distance_km": km, "elevation_difference_m": dz,\n                                "pressure_compatible": a["pressure_datum"] == b["pressure_datum"],\n                                "neighbor_role": b["station_role"]})\n        edges.extend(sorted(options, key=lambda x: x["distance_km"])[:max_neighbors])\n    return pd.DataFrame(edges, columns=["station_id", "neighbor_id", "distance_km", "elevation_difference_m", "pressure_compatible", "neighbor_role"])\n\n\ndef prepare(root):\n    root = Path(root)\n    catalog = pd.read_csv(root / "station_catalog.csv", dtype={"station_id": str})\n    rows = []\n    processed = root / "processed"\n    processed.mkdir(exist_ok=True)\n    code_hash = digest(Path(__file__))\n    for meta in catalog.to_dict("records"):\n        paths = [root / f"raw/{y}/{meta[\'station_id\']}.csv" for y in YEARS]\n        paths = [p for p in paths if p.exists() and p.with_name(p.name + ".receipt.json").exists()]\n        if not paths:\n            continue\n        raw_hashes = {str(p.relative_to(root)): digest(p) for p in paths}\n        target = processed / f"{meta[\'station_id\']}.parquet"\n        cached_receipt = target.with_name(target.name + ".profile.json")\n        if target.exists() and cached_receipt.exists():\n            cached = json.loads(cached_receipt.read_text())\n            if cached.get("code_hash") == code_hash and cached.get("raw_hashes") == raw_hashes and cached["record"]["processed_sha256"] == digest(target):\n                rows.append(cached["record"])\n                continue\n        data = []\n        for path in paths:\n            if digest(path) != json.loads(path.with_name(path.name + ".receipt.json").read_text())["sha256"]:\n                raise ValueError(f"Corrupt raw cache: {path}")\n            raw = pd.read_csv(path, dtype=str, low_memory=False)\n            dates = pd.to_datetime(raw.DATE, utc=True, errors="coerce")\n            if dates.dropna().dt.year.ne(int(path.parent.name)).any():\n                raise ValueError(f"Unexpected year inside {path}; source schema review required")\n            data.append(normalize(raw, meta))\n        frame, counts = choose_pressure(pd.concat(data, ignore_index=True).sort_values("timestamp_utc"))\n        frame = frame.drop_duplicates(["station_id", "timestamp_utc"]).reset_index(drop=True)\n        train = profile(frame, "fit")\n        eligible = train["screened_proxy_rows"] >= 1000 and train["screened_days"] >= 180\n        record = {**{k: meta[k] for k in ["station_id", "station_name", "latitude", "longitude", "elevation_m", "station_role", "geographic_block", "legacy_24_station"]},\n                  "pressure_datum": frame.pressure_datum.iloc[0] if len(frame) else "unavailable",\n                  "training_eligible": eligible, "exclusion_reason": "" if eligible else "fewer_than_1000_screened_rows_or_180_days_in_2020_21",\n                  **{"fit_" + k: v for k, v in train.items()}, **{"eval_" + k: v for k, v in profile(frame, "eval").items()},\n                  **{"fit_count_" + k: v for k, v in counts.items()}, "raw_files": len(paths)}\n        tmp = target.with_suffix(".partial.parquet")\n        frame.to_parquet(tmp, index=False)\n        atomic_replace(tmp, target)\n        record["processed_sha256"] = digest(target)\n        write_json(cached_receipt, {"code_hash": code_hash, "raw_hashes": raw_hashes, "record": record})\n        rows.append(record)\n        print(f"Prepared {meta[\'station_id\']}: {len(frame):,} rows; training eligible={eligible}", flush=True)\n    summary = pd.DataFrame(rows)\n    if summary.empty:\n        raise ValueError("No downloaded station files. Run acquisition first.")\n    summary.to_csv(root / "station_quality.csv", index=False)\n    ready = summary.loc[summary.training_eligible]\n    graph = neighbor_graph(ready)\n    graph.to_csv(root / "neighbor_graph.csv", index=False)\n    support = ready[["station_id", "station_role", "pressure_datum", "legacy_24_station"]].copy()\n    support["geographic_buddies"] = support.station_id.map(graph.groupby("station_id").size()).fillna(0).astype(int)\n    support["pressure_compatible_buddies"] = support.station_id.map(graph.loc[graph.pressure_compatible].groupby("station_id").size()).fillna(0).astype(int)\n    support.to_csv(root / "spatial_support.csv", index=False)\n    receipt = {"version": VERSION, "created_at_utc": utc_now(), "downloaded_stations": len(summary),\n               "training_eligible_stations": len(ready), "new_eligible_outside_old24": int((~ready.legacy_24_station).sum()),\n               "fit_screened_proxy_rows": int(ready.fit_screened_proxy_rows.sum()),\n               "stations_with_at_least_2_geographic_buddies": int(support.geographic_buddies.ge(2).sum()),\n               "stations_with_at_least_2_pressure_buddies": int(support.pressure_compatible_buddies.ge(2).sum()),\n               "caveat": "Geographic support is an upper bound; contemporaneous non-stale readings also required",\n               "ready_for_expanded_training": bool(len(ready) >= 50 and (~ready.legacy_24_station).sum() >= 25),\n               "no_verified_real_fault_labels": True, "pressure_contract": "one type per station fitted on 2020-21; missing retained, no fallback",\n               "all_data_hash": hashlib.sha256(summary[["station_id", "processed_sha256"]].sort_values("station_id").to_csv(index=False).encode()).hexdigest()}\n    write_json(root / "data_readiness.json", receipt)\n    return summary, graph, receipt\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("action", choices=["discover", "download", "prepare"])\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--benchmark", type=Path)\n    parser.add_argument("--stations", nargs="*")\n    args = parser.parse_args()\n    if args.action == "discover":\n        old = pd.read_csv(args.benchmark, dtype=str).station_id if args.benchmark else []\n        print(discover(args.root, old)[2])\n    elif args.action == "download":\n        print(acquire(args.root, args.stations).status.value_counts())\n    else:\n        print(prepare(args.root)[2])\n', 'iteration11_training': '"""Bounded-memory, causal research baselines for the Iteration 11 data rebuild.\n\nNOT operational IMD validation. Fault/weather labels below are synthetic and\npresumed-normal source-QC labels are imperfect. No promotion to live models.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.metadata\nimport json\nfrom functools import lru_cache\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss,\n                             confusion_matrix, f1_score, precision_score, recall_score)\n\nfrom iteration11_data import PRIMARY, VERSION, atomic_replace, digest, write_json\n\nFEATURES = ["gap_minutes", "missing_count"]\nfor prefix in ["temperature", "pressure", "humidity"]:\n    FEATURES += [f"{prefix}_{suffix}" for suffix in [\n        "delta", "rate", "z24", "iqr24", "frozen_minutes", "slope3h", "slope6h", "slope24h",\n        "buddy_count", "buddy_residual", "buddy_spread", "buddy_age_minutes", "buddy_trend_residual"]]\nFEATURES += ["hour_sin", "hour_cos", "year_sin", "year_cos"]\nSPATIAL_FEATURES = [x for x in FEATURES if "buddy_" in x]\nFAULT_TYPES = ["spike", "bias", "drift", "frozen", "noise", "missing_value"]\n\n\nclass RobustQC:\n    """Non-learning temporal QC comparator; output is a score until calibrated."""\n    def predict_proba(self, frame):\n        z = frame[[f"{p}_z24" for p in ["temperature", "pressure", "humidity"]]].abs().max(axis=1).fillna(0)\n        score = np.maximum(z.to_numpy(), frame.missing_count.to_numpy() * 10)\n        p = 1 / (1 + np.exp(-np.clip(score - 6, -30, 30)))\n        return np.column_stack([1-p, p])\n\n\ndef seed_for(text, seed):\n    return int(hashlib.sha256(f"{seed}:{text}".encode()).hexdigest()[:8], 16)\n\n\ndef inject(frame, seed=111):\n    """New episodes, not inherited old benchmark labels. Injection BEFORE features.\n\n    All buddies get the SAME deterministic simulator, including their faults;\n    no clean-reference/oracle neighbor stream. Holdout labels are simulator-only.\n    """\n    f = frame.sort_values("timestamp_utc").reset_index(drop=True).copy()\n    f["y"] = np.where(f.qc_screened_proxy, 0, -1)\n    f["fault_type"] = ""\n    f["episode_id"] = ""\n    f["weather_challenge"] = False\n    if f.empty:\n        return f\n    sid = f.station_id.iloc[0]\n    # Coherent, smooth 24h T/P/RH perturbations for same geographic block.\n    # Synthetic challenges, NOT verified cyclones/fronts or meteorological labels.\n    for year in sorted(f.timestamp_utc.dt.year.unique()):\n        block_rng = np.random.default_rng(seed_for(f"weather:{f.geographic_block.iloc[0]}:{year}", seed))\n        for month in [2, 5, 8, 11]:\n            start = pd.Timestamp(year=int(year), month=month, day=int(block_rng.integers(8, 15)), tz="UTC")\n            phase = (f.timestamp_utc - start).dt.total_seconds() / 86400\n            mask = phase.between(0, 1) & f.y.eq(0)\n            bump = np.sin(np.pi * phase[mask]) ** 2\n            for column, size in zip(PRIMARY, [-4.0, -5.0, 12.0]):\n                f.loc[mask, column] += size * bump\n            f.loc[mask, "relative_humidity_pct"] = f.loc[mask, "relative_humidity_pct"].clip(0, 100)\n            f.loc[mask, "weather_challenge"] = True\n    for year in sorted(f.timestamp_utc.dt.year.unique()):\n        rng = np.random.default_rng(seed_for(f"fault:{sid}:{year}", seed))\n        for month in range(1, 13):\n            # Two episodes / month, rotating faults; keep real cadence and gaps.\n            for slot in range(2):\n                kind = FAULT_TYPES[(month * 2 + slot) % len(FAULT_TYPES)]\n                possible = f.index[(f.timestamp_utc.dt.year == year) & (f.timestamp_utc.dt.month == month)\n                                   & f.y.eq(0) & ~f.weather_challenge & f.episode_id.eq("")]\n                if len(possible) < 20:\n                    continue\n                begin = int(rng.choice(possible[3:-3]))\n                hours = 0 if kind == "spike" else float(rng.choice([3, 6, 12, 24]))\n                until = f.timestamp_utc.iloc[begin] + pd.Timedelta(hours=hours)\n                ids = f.index[(f.index >= begin) & (f.timestamp_utc <= until) & f.y.eq(0)\n                              & ~f.weather_challenge & f.episode_id.eq("")]\n                if not len(ids):\n                    continue\n                # Do not join an injected episode across long natural reporting gaps.\n                gap = f.loc[ids, "timestamp_utc"].diff().dt.total_seconds().gt(6 * 3600)\n                if gap.any():\n                    ids = ids[:int(np.flatnonzero(gap)[0])]\n                if len(ids) < 2 and kind not in {"spike", "missing_value"}:\n                    continue\n                col_index = int(rng.integers(0, 3))\n                col = PRIMARY[col_index]\n                amp = [3.0, 5.0, 12.0][col_index] * float(rng.uniform(.5, 2)) * rng.choice([-1, 1])\n                if kind == "spike":\n                    f.loc[ids, col] += amp * 2\n                elif kind == "bias":\n                    f.loc[ids, col] += amp\n                elif kind == "drift":\n                    f.loc[ids, col] += np.linspace(amp * .1, amp, len(ids))\n                elif kind == "frozen":\n                    value = f.loc[max(0, begin - 1), col]\n                    if not np.isfinite(value) or np.allclose(f.loc[ids, col], value):\n                        continue\n                    f.loc[ids, col] = value\n                elif kind == "noise":\n                    f.loc[ids, col] += rng.normal(0, abs(amp), len(ids))\n                elif kind == "missing_value":\n                    f.loc[ids, col] = np.nan\n                f.loc[ids, "y"] = 1\n                f.loc[ids, "fault_type"] = kind\n                f.loc[ids, "episode_id"] = f"{seed}:{sid}:{year}:{month}:{slot}"\n    return f\n\n\ndef temporal(frame):\n    f = frame.sort_values("timestamp_utc").reset_index(drop=True)\n    out = pd.DataFrame(index=f.index)\n    stamps = pd.DatetimeIndex(f.timestamp_utc)\n    minutes = f.timestamp_utc.diff().dt.total_seconds().div(60)\n    out["gap_minutes"] = minutes\n    out["missing_count"] = f[PRIMARY].isna().sum(axis=1)\n    # All rolling baselines closed=\'left\': current/future values never enter.\n    hours = pd.Series((stamps - stamps[0]).total_seconds() / 3600, index=stamps)\n    for col, prefix, floor in zip(PRIMARY, ["temperature", "pressure", "humidity"], [.3, .5, 2.0]):\n        s = pd.Series(f[col].to_numpy(float), index=stamps)\n        roll = s.rolling("24h", closed="left", min_periods=4)\n        med = roll.median()\n        spread = (roll.quantile(.75) - roll.quantile(.25)).clip(lower=floor)\n        out[prefix + "_delta"] = s.diff().to_numpy()\n        out[prefix + "_rate"] = (s.diff().to_numpy() / minutes.clip(lower=1).to_numpy() * 60)\n        out[prefix + "_z24"] = ((s - med) / spread).to_numpy()\n        out[prefix + "_iqr24"] = spread.to_numpy()\n        changes = f[col].ne(f[col].shift()) | minutes.gt(360) | f[col].isna()\n        run_start = f.timestamp_utc.groupby(changes.cumsum()).transform("first")\n        out[prefix + "_frozen_minutes"] = (f.timestamp_utc - run_start).dt.total_seconds().div(60)\n        for window in [3, 6, 24]:\n            valid_hours = hours.where(s.notna())\n            k = dict(window=f"{window}h", closed="left", min_periods=3)\n            cov = (valid_hours * s).rolling(**k).mean() - valid_hours.rolling(**k).mean() * s.rolling(**k).mean()\n            var = (valid_hours ** 2).rolling(**k).mean() - valid_hours.rolling(**k).mean() ** 2\n            out[f"{prefix}_slope{window}h"] = (cov / var.where(var > 1e-6)).to_numpy()\n    hr = stamps.hour + stamps.minute / 60\n    out["hour_sin"], out["hour_cos"] = np.sin(2*np.pi*hr/24), np.cos(2*np.pi*hr/24)\n    out["year_sin"], out["year_cos"] = np.sin(2*np.pi*stamps.dayofyear/365.25), np.cos(2*np.pi*stamps.dayofyear/365.25)\n    return out.replace([np.inf, -np.inf], np.nan)\n\n\ndef spatial(frame, temp, buddies, max_age_minutes=90):\n    """Past-only matches. Pressure compared only within compatible datums.\n\n    Differences are temporal residuals, not absolute raw pressure across heights.\n    Scores abstain (NaN) with fewer than two independent valid buddies.\n    """\n    f = frame.sort_values("timestamp_utc").reset_index(drop=True)\n    result = temp.copy()\n    for prefix in ["temperature", "pressure", "humidity"]:\n        vals, trends, ages = [], [], []\n        for other, features, compatible in buddies:\n            if prefix == "pressure" and not compatible:\n                continue\n            right = pd.DataFrame({"neighbor_time": other.timestamp_utc.to_numpy(),\n                                  "z": features[prefix + "_z24"].to_numpy(),\n                                  "trend": features[prefix + "_slope6h"].to_numpy()}).sort_values("neighbor_time")\n            matched = pd.merge_asof(f[["timestamp_utc"]], right, left_on="timestamp_utc", right_on="neighbor_time",\n                                    direction="backward", tolerance=pd.Timedelta(minutes=max_age_minutes))\n            vals.append(matched.z)\n            trends.append(matched.trend)\n            ages.append((matched.timestamp_utc - matched.neighbor_time).dt.total_seconds().div(60).where(matched.z.notna()))\n        if vals:\n            v = pd.concat(vals, axis=1)\n            t = pd.concat(trends, axis=1)\n            a = pd.concat(ages, axis=1)\n            count = v.notna().sum(axis=1)\n            center = v.median(axis=1)\n            supported = count.ge(2)\n            result[prefix + "_buddy_count"] = count\n            result[prefix + "_buddy_residual"] = (temp[prefix + "_z24"] - center).where(supported)\n            result[prefix + "_buddy_spread"] = v.sub(center, axis=0).abs().median(axis=1).where(supported)\n            result[prefix + "_buddy_age_minutes"] = a.max(axis=1).where(supported)\n            result[prefix + "_buddy_trend_residual"] = (temp[prefix + "_slope6h"] - t.median(axis=1)).where(t.notna().sum(axis=1).ge(2))\n        else:\n            for suffix in ["count", "residual", "spread", "age_minutes", "trend_residual"]:\n                result[f"{prefix}_buddy_{suffix}"] = 0 if suffix == "count" else np.nan\n    return result[FEATURES].replace([np.inf, -np.inf], np.nan).astype(np.float32)\n\n\ndef split_roles(times, station_role):\n    t = pd.to_datetime(times, utc=True)\n    year = t.dt.year\n    split = pd.Series("unused", index=t.index)\n    if station_role == "spatial_holdout":\n        split.loc[year.eq(2023)] = "spatial_confirmation"\n        return split\n    split.loc[year.le(2021)] = "train"\n    split.loc[year.eq(2022) & t.dt.month.le(4)] = "early_stop"\n    split.loc[year.eq(2022) & t.dt.month.between(5, 8)] = "calibration"\n    split.loc[year.eq(2022) & t.dt.month.ge(9)] = "policy"\n    split.loc[year.eq(2023)] = "temporal_confirmation"\n    return split\n\n\ndef build_features(root, seed=111, min_ready=True):\n    root = Path(root)\n    readiness = json.loads((root / "data_readiness.json").read_text())\n    if min_ready and not readiness["ready_for_expanded_training"]:\n        raise RuntimeError("Data gate failed: need >=50 eligible stations, >=25 outside legacy24. Download/profile first.")\n    quality = pd.read_csv(root / "station_quality.csv", dtype={"station_id": str})\n    ready = quality.loc[quality.training_eligible].set_index("station_id")\n    graph = pd.read_csv(root / "neighbor_graph.csv", dtype={"station_id": str, "neighbor_id": str})\n    target = root / f"features_seed{seed}"\n    target.mkdir(exist_ok=True)\n    contract = {"version": VERSION, "source_sha256": digest(Path(__file__)), "data_hash": readiness["all_data_hash"],\n                "seed": seed, "features": FEATURES, "no_future_neighbors": True,\n                "training_neighbors": "development stations only; no spatial holdout contamination",\n                "confirmation_neighbors": "all eligible contemporaneous streams, with faults injected; not clean oracles",\n                "labels": "synthetic faults and coherent perturbations on QC-screened presumed-normal proxy",\n                "2023": "development confirmation, previously used elsewhere; NOT a new blind test"}\n    contract_path = target / "contract.json"\n    if contract_path.exists() and json.loads(contract_path.read_text()) != contract:\n        raise RuntimeError("Feature contract changed: use a new root/version; stale caches will not be reused")\n    write_json(contract_path, contract)\n    @lru_cache(maxsize=10)\n    def stream(sid):\n        path = root / f"processed/{sid}.parquet"\n        if digest(path) != ready.loc[sid, "processed_sha256"]:\n            raise ValueError(f"Processed hash mismatch: {sid}")\n        raw = pd.read_parquet(path)\n        # per-year feature boundaries intentionally reset, no next-year influence\n        f = inject(raw, seed)\n        pieces = [temporal(g.reset_index(drop=True)) for _, g in f.groupby(f.timestamp_utc.dt.year, sort=True)]\n        return f, pd.concat(pieces, ignore_index=True)\n    for sid in ready.index:\n        dest = target / f"{sid}.parquet"\n        receipt_path = dest.with_name(dest.name + ".json")\n        if dest.exists() and receipt_path.exists() and digest(dest) == json.loads(receipt_path.read_text())["sha256"]:\n            continue\n        frame, temp = stream(sid)\n        parts = []\n        for year, group in frame.groupby(frame.timestamp_utc.dt.year, sort=True):\n            take = group.index\n            own = group.reset_index(drop=True)\n            own_temp = temp.loc[take].reset_index(drop=True)\n            neighbours = []\n            for edge in graph.loc[graph.station_id.eq(sid)].to_dict("records"):\n                if year <= 2022 and edge["neighbor_role"] != "development":\n                    continue\n                neighbor, nf = stream(edge["neighbor_id"])\n                ids = neighbor.index[neighbor.timestamp_utc.dt.year.eq(year)]\n                if len(ids):\n                    neighbours.append((neighbor.loc[ids].reset_index(drop=True), nf.loc[ids].reset_index(drop=True), bool(edge["pressure_compatible"])))\n            feats = spatial(own, own_temp, neighbours)\n            for col in ["station_id", "timestamp_utc", "y", "fault_type", "episode_id", "weather_challenge", "geographic_block"]:\n                feats[col] = own[col].to_numpy()\n            feats["split"] = split_roles(own.timestamp_utc, ready.loc[sid, "station_role"]).to_numpy()\n            feats["legacy_24_station"] = bool(ready.loc[sid, "legacy_24_station"])\n            parts.append(feats)\n        combined = pd.concat(parts, ignore_index=True)\n        tmp = dest.with_suffix(".partial.parquet")\n        combined.to_parquet(tmp, index=False)\n        atomic_replace(tmp, dest)\n        write_json(receipt_path, {"sha256": digest(dest), "rows": len(combined)})\n        print(f"Features: {sid}, {len(combined):,} rows", flush=True)\n    stream.cache_clear()\n    return target\n\n\ndef load_partition(directory, split, negative_cap=1500, seed=11, legacy_only=False):\n    """Per-station deterministic negative subsampling; never samples final metrics."""\n    rows = []\n    for path in sorted(Path(directory).glob("*.parquet")):\n        f = pd.read_parquet(path)\n        f = f.loc[f.split.eq(split) & f.y.ge(0)]\n        if legacy_only:\n            f = f.loc[f.legacy_24_station]\n        if negative_cap is not None:\n            pos = f.loc[f.y.eq(1) | f.weather_challenge]\n            neg = f.loc[f.y.eq(0) & ~f.weather_challenge]\n            if len(neg) > negative_cap:\n                neg = neg.sample(negative_cap, random_state=seed_for(str(path.name), seed))\n            f = pd.concat([pos, neg])\n        rows.append(f)\n    if not rows:\n        raise ValueError("Feature directory is empty")\n    return pd.concat(rows, ignore_index=True)\n\n\ndef weights(frame):\n    # Equal station mass, with capped episode weighting inside each station.\n    w = 1 / frame.station_id.map(frame.station_id.value_counts()).to_numpy(float)\n    positive = frame.y.eq(1).to_numpy()\n    if positive.any():\n        ep = frame.loc[positive, "episode_id"]\n        w[positive] /= np.sqrt(ep.map(ep.value_counts()).to_numpy(float))\n    return w / w.mean()\n\n\ndef metrics(frame, probability, threshold):\n    y = frame.y.to_numpy(int)\n    pred = np.asarray(probability) >= threshold\n    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()\n    day = frame.timestamp_utc.dt.strftime("%Y-%m-%d")\n    station_days = (frame.station_id + ":" + day).nunique()\n    episodic = pd.DataFrame({"ep": frame.episode_id.to_numpy(), "hit": pred})\n    episodic = episodic.loc[episodic.ep.ne("")].groupby("ep").hit.max()\n    weather = frame.weather_challenge.to_numpy(bool) & (y == 0)\n    return {"rows": len(y), "positives": int(y.sum()), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),\n            "precision": float(precision_score(y, pred, zero_division=0)), "recall": float(recall_score(y, pred, zero_division=0)),\n            "f1": float(f1_score(y, pred, zero_division=0)), "accuracy": float(accuracy_score(y, pred)),\n            "pr_auc_ap": float(average_precision_score(y, probability)) if y.sum() else None,\n            "brier": float(brier_score_loss(y, probability)),\n            "fault_episode_recall": float(episodic.mean()) if len(episodic) else None,\n            "false_positive_rows_per_observed_station_day": float(fp / max(station_days, 1)),\n            "synthetic_weather_false_alarm_fraction": float(pred[weather].mean()) if weather.any() else None}\n\n\ndef calibrated(model, x, calibrator):\n    p = np.clip(model.predict_proba(x)[:, 1], 1e-6, 1-1e-6)\n    return calibrator.predict_proba(np.log(p/(1-p)).reshape(-1, 1))[:, 1]\n\n\ndef score_partition(directory, split, model, features, calibrator=None):\n    """Keep only labels/timestamps/probabilities in RAM for full-prevalence scoring."""\n    rows = []\n    for path in sorted(Path(directory).glob("*.parquet")):\n        f = pd.read_parquet(path)\n        f = f.loc[f.split.eq(split) & f.y.ge(0)]\n        if f.empty:\n            continue\n        p = (calibrated(model, f[features], calibrator) if calibrator is not None\n             else model.predict_proba(f[features])[:, 1])\n        slim = f[["station_id", "timestamp_utc", "y", "episode_id", "weather_challenge", "fault_type"]].copy()\n        slim["probability"] = p\n        rows.append(slim)\n    if not rows:\n        raise RuntimeError(f"No observations in {split}")\n    return pd.concat(rows, ignore_index=True)\n\n\ndef train_research(root, seed=111, use_catboost=False, gpu=True, smoke=False):\n    from lightgbm import LGBMClassifier, early_stopping, log_evaluation\n    root = Path(root)\n    features_dir = root / f"features_seed{seed}"\n    output = root / f"research_seed{seed}"\n    if (output / "frozen_policies.json").exists():\n        return output  # no implicit repeated model selection or overwriting frozen candidates\n    output.mkdir(exist_ok=True)\n    fit = load_partition(features_dir, "train", negative_cap=2500)\n    early = load_partition(features_dir, "early_stop", negative_cap=1500)\n    if set(fit.y.unique()) != {0, 1} or set(early.y.unique()) != {0, 1}:\n        raise RuntimeError("Both classes required in fit and early-stop periods")\n    candidates = [("lightgbm_temporal", [x for x in FEATURES if x not in SPATIAL_FEATURES], False),\n                  ("lightgbm_spatial", FEATURES, False)]\n    if fit.loc[fit.legacy_24_station, "station_id"].nunique() >= 5:\n        candidates.append(("lightgbm_legacy24_retrained", FEATURES, True))\n    if use_catboost:\n        candidates.append(("catboost_spatial", FEATURES, False))\n    models = [("robust_qc", RobustQC(), FEATURES, False)]\n    for name, feats, legacy in candidates:\n        sub = fit.loc[fit.legacy_24_station] if legacy else fit\n        if name.startswith("catboost"):\n            from catboost import CatBoostClassifier\n            model = CatBoostClassifier(iterations=60 if smoke else 1400, depth=6, learning_rate=.045,\n                                       loss_function="Logloss", eval_metric="AUC", l2_leaf_reg=12,\n                                       random_seed=seed, task_type="GPU" if gpu else "CPU", verbose=False,\n                                       allow_writing_files=False, thread_count=4)\n            model.fit(sub[feats], sub.y, sample_weight=weights(sub), eval_set=(early[feats], early.y),\n                      early_stopping_rounds=100, verbose=False)\n        else:\n            model = LGBMClassifier(n_estimators=60 if smoke else 1400, num_leaves=31, max_depth=-1,\n                                  learning_rate=.04, min_child_samples=120, reg_alpha=1, reg_lambda=12,\n                                  colsample_bytree=.85, subsample=.85, subsample_freq=1,\n                                  random_state=seed, n_jobs=4, verbosity=-1, force_col_wise=True,\n                                  metric="average_precision")\n            model.fit(sub[feats], sub.y, sample_weight=weights(sub), eval_set=[(early[feats], early.y)],\n                      eval_metric="average_precision", callbacks=[early_stopping(100, first_metric_only=True, verbose=False), log_evaluation(0)])\n        models.append((name, model, feats, legacy))\n        print("Trained", name, "on", len(sub), "sampled rows from", sub.station_id.nunique(), "stations", flush=True)\n    del fit, early\n    # Full-prevalence calibration and policy sets, distinct four-month blocks.\n    # If the full sets exceed RAM, score streaming by station instead of subsampling negatives.\n    bundles = []\n    for name, model, feats, legacy in models:\n        cal = score_partition(features_dir, "calibration", model, feats)\n        if set(cal.y.unique()) != {0, 1}:\n            raise RuntimeError("Calibration block needs both classes")\n        p = np.clip(cal.probability.to_numpy(), 1e-6, 1-1e-6)\n        calibrator = LogisticRegression(C=1, solver="lbfgs", max_iter=500)\n        calibrator.fit(np.log(p/(1-p)).reshape(-1, 1), cal.y)\n        bundles.append({"name": name, "model": model, "features": feats, "calibrator": calibrator,\n                        "legacy_retrained": legacy, "research_only": True})\n    del cal\n    frontier, decisions = [], []\n    for bundle in bundles:\n        policy = score_partition(features_dir, "policy", bundle["model"], bundle["features"], bundle["calibrator"])\n        if policy.y.sum() == 0:\n            raise RuntimeError("Policy period has no injected positives")\n        p = policy.probability.to_numpy()\n        rows = []\n        for th in sorted(set([1.000001, *np.geomspace(.0001, .1, 24), *np.linspace(.1, .99, 35)])):\n            item = {"model": bundle["name"], "threshold": float(th), **metrics(policy, p, th)}\n            item["meets_development_budget"] = bool(item["precision"] >= .8 and item["false_positive_rows_per_observed_station_day"] <= .05)\n            rows.append(item)\n        acceptable = [x for x in rows if x["meets_development_budget"]]\n        # No gate passing != operationally acceptable. Freeze a descriptive best-F1 comparator only.\n        chosen = max(acceptable or rows, key=lambda x: x["f1"])\n        bundle["threshold"] = chosen["threshold"]\n        bundle["policy_gate_passed"] = bool(acceptable)\n        model_path = output / (bundle["name"] + ".joblib")\n        joblib.dump(bundle, model_path)\n        decisions.append({**chosen, "path": model_path.name, "sha256": digest(model_path)})\n        frontier.extend(rows)\n    pd.DataFrame(frontier).to_csv(output / "policy_frontier.csv", index=False)\n    write_json(output / "frozen_policies.json", {"version": VERSION, "seed": seed, "smoke_test_only": smoke,\n        "feature_contract_sha256": digest(features_dir / "contract.json"), "policies": decisions,\n        "warning": "Not promoted; 2023 confirmation is NOT fresh blind evidence; thresholds must not be retuned there",\n        "versions": {p: importlib.metadata.version(p) for p in ["numpy", "pandas", "scikit-learn", "lightgbm", "joblib"]}})\n    return output\n\n\ndef confirm_research(root, seed=111):\n    root = Path(root)\n    output = root / f"research_seed{seed}"\n    result_path = output / "iteration11_result_block.json"\n    if result_path.exists():\n        return json.loads(result_path.read_text())\n    lock = json.loads((output / "frozen_policies.json").read_text())\n    if digest(root / f"features_seed{seed}/contract.json") != lock["feature_contract_sha256"]:\n        raise RuntimeError("Feature contract differs from frozen training input")\n    comparison, per_station, fault_rows = [], [], []\n    # Read only station partitions, so full network scoring has bounded memory.\n    for item in lock["policies"]:\n        path = output / item["path"]\n        if digest(path) != item["sha256"]:\n            raise RuntimeError("Model hash mismatch")\n        bundle = joblib.load(path)  # only artifacts generated by this pipeline\n        grouped = {"temporal_confirmation": [], "spatial_confirmation": []}\n        for part in sorted((root / f"features_seed{seed}").glob("*.parquet")):\n            f = pd.read_parquet(part)\n            f = f.loc[f.split.isin(grouped) & f.y.ge(0)]\n            if f.empty:\n                continue\n            prob = calibrated(bundle["model"], f[bundle["features"]], bundle["calibrator"])\n            slim = f[["station_id", "timestamp_utc", "y", "fault_type", "episode_id", "weather_challenge", "split"]].copy()\n            slim["probability"] = prob\n            for scope, g in slim.groupby("split"):\n                grouped[scope].append(g)\n                per_station.append({"model": bundle["name"], "split": scope, "station_id": part.stem,\n                                    **metrics(g, g.probability, bundle["threshold"])})\n        for scope, frames in grouped.items():\n            if not frames:\n                continue\n            frame = pd.concat(frames, ignore_index=True)\n            comparison.append({"model": bundle["name"], "split": scope, **metrics(frame, frame.probability, bundle["threshold"])})\n            for kind, g in frame.loc[frame.y.eq(1)].groupby("fault_type"):\n                pred = g.probability.ge(bundle["threshold"])\n                eps = pd.DataFrame({"id": g.episode_id, "hit": pred}).groupby("id").hit.max()\n                fault_rows.append({"model": bundle["name"], "split": scope, "fault": kind,\n                                   "point_recall": float(pred.mean()), "episode_recall": float(eps.mean()),\n                                   "positive_rows": len(g), "episodes": len(eps)})\n    pd.DataFrame(comparison).to_csv(output / "iteration11_comparison.csv", index=False)\n    pd.DataFrame(per_station).to_csv(output / "iteration11_station_metrics.csv", index=False)\n    pd.DataFrame(fault_rows).to_csv(output / "iteration11_fault_recall.csv", index=False)\n    result = {"version": VERSION, "seed": seed, "smoke_test_only": lock["smoke_test_only"],\n              "data_readiness": json.loads((root / "data_readiness.json").read_text()), "comparison": comparison,\n              "live_model_changed": False, "real_sensor_fault_accuracy": "unknown: no verified field fault labels",\n              "unimplemented_in_this_baseline": ["packet loss detection without heartbeat contract", "validated maintenance prediction",\n                                                 "root cause classification", "automatic correction", "IMD authorization/telemetry adapter"],\n              "status": "Research data-expansion comparison; requires review, event bootstrap uncertainty and external AWS validation before promotion"}\n    write_json(result_path, result)\n    return result\n'}
EXPECTED_MODULE_SHA256 = {'iteration11_data': '07bb3aa937a351d8890e4c3434ad9b76d6ae466bf2b87584896c08ee4f7dbfcc', 'iteration11_training': '0bb7a3522b5f4e1b3e5bb491e7cac3aa359eb0fc6c0758afa70bfe8fb5c3e172'}
import hashlib, importlib
MODULE_ROOT = Path('/content/skyguard_iteration11_rebuild_code')
MODULE_ROOT.mkdir(exist_ok=True)
for name, text in MODULE_SOURCES.items():
    assert hashlib.sha256(text.encode()).hexdigest() == EXPECTED_MODULE_SHA256[name]
    (MODULE_ROOT / (name + '.py')).write_text(text, encoding='utf-8')
sys.path.insert(0, str(MODULE_ROOT)) if str(MODULE_ROOT) not in sys.path else None
import iteration11_data as i11d
import iteration11_training as i11t
importlib.reload(i11d)
importlib.reload(i11t)
print('Code loaded:', i11d.VERSION, '| explicit feature count:', len(i11t.FEATURES))

## 3. Source discovery — metadata is not observation coverage

All Indian ISD IDs are considered, not a hardcoded24-station selection. Annual directory listings verify
which historical CSVs exist. File downloads then verify schema and hashes. Missing/incompatible files
are reported instead of replaced with synthetic observations. NOAA is transitioning from ISD to GHCNh;
this notebook deliberately supports the verified **historical ISD CSV schema only**, with no silent format fallback.

Sources:
- [NOAA station histories](https://www.ncei.noaa.gov/products/land-based-station/station-histories)
- [ISD data and limitations](https://www.ncei.noaa.gov/products/land-based-station/integrated-surface-database)
- [GHCNh transition](https://www.ncei.noaa.gov/products/global-historical-climatology-network-hourly)
- [IMD AWS API](https://api.imd.gov.in/public/api_reference.html)
- [Official 1008 IMD AWS count](https://www.pib.gov.in/PressReleasePage.aspx?PRID=2241702)

In [ ]:
LEGACY_24_IDS = ['42181099999', '42182099999', '42348099999', '42189099999', '42361099999', '42131099999', '43128099999', '43128599999', '43181099999', '43021099999', '43213099999', '43086099999', '43295099999', '43302599999', '42705699999', '43321099999', '43284099999', '43233099999', '43279099999', '43278099999', '43275099999', '43329099999', '43245099999', '43331099999']
catalog, download_plan, discovery = i11d.discover(I11_ROOT, LEGACY_24_IDS)
display(pd.Series(discovery))
display(catalog[['station_id','station_name','latitude','longitude','station_role',
                 'inventory_reports_2020','inventory_reports_2021','legacy_24_station']].head(15))
print('Eligible station count will be computed AFTER downloading and inspecting actual observations.')

## 4. Download real observations, 2020–2023

Requests are limited to three simultaneous downloads with retries, disk checks and per-file receipts.
No IMD credentials are required for this public proxy source. This does **not** obtain authorized IMD AWS data.
Do not count map/catalog entries as downloaded stations. No 2024/2025 labelled test files are opened.

In [ ]:
PILOT_IDS = ['42369099999','42372099999','42273099999','42492099999','42591099999','42391099999']
selected_ids = None if RUN_MODE == 'all_available' else PILOT_IDS
download_status = i11d.acquire(I11_ROOT, selected_ids, workers=3)
display(download_status.status.value_counts())
failed_downloads = download_status.loc[download_status.status.ne('complete')]
if len(failed_downloads):
    display(failed_downloads[['station_id','year','error']])
    print('Rerun this cell to retry. Incomplete acquisition cannot silently become a full-network experiment.')
print('Completed unique stations in this requested run:', download_status.loc[download_status.status.eq('complete'),'station_id'].nunique())

## 5. Normalize and profile — no pressure-type mixing

Each station preserves station pressure, mean-sea-level pressure and altimeter pressure separately.
One pressure type is selected from usable **2020–2021 data only** and kept fixed; absent values stay missing.
RH is calculated from reported T/dew point without silently clipping invalid results. Source quality flags
remain audit fields, **never model predictors or verified fault labels**. Duplicates use deterministic quality/completeness ordering.

Training eligibility: >=1,000 QC-screened triples across >=180 distinct 2020–2021 days.
QC-screened means **presumed normal, not confirmed healthy**; other rows stay unknown and are not negative labels.
Pressure datum and sensor-resolution differences can cause spurious alarms even with many stations.

In [ ]:
station_quality, graph, readiness = i11d.prepare(I11_ROOT)
display(pd.Series(readiness))
display(station_quality[['station_id','station_name','training_eligible','pressure_datum',
                        'fit_rows','fit_screened_proxy_rows','fit_screened_days','fit_median_cadence_minutes','exclusion_reason']])
display(pd.read_csv(I11_ROOT/'spatial_support.csv', dtype={'station_id':str}).head(30))

## 6. Data gate and split contract

| Purpose | Observations | Station scope |
|---|---|---|
| Fit models | 2020–2021 | Development geographic blocks |
| Early stopping | January–April 2022 | Development blocks |
| Probability calibration | May–August 2022 | Development blocks, unsampled prevalence |
| Threshold selection | September–December 2022 | Development blocks, unsampled prevalence |
| Temporal confirmation | 2023 | Development blocks |
| Spatial confirmation | 2023 | Held-out geographic blocks |

Spatial holdouts supply no training examples and no pre-2023 training buddies. Their historical unlabelled
data selects their own pressure type/eligibility: this is **label-held-out station transfer with historical context**,
not zero-history cold-start performance. 2023 has been inspected elsewhere in this project; do not call it blind.
No failed false-alarm gate is turned into an operational promotion.

Small pilots stop here for data review. They can export a report in the last cell without training.

In [ ]:
all_files_complete = download_status.status.eq('complete').all()
CAN_TRAIN = bool(RUN_MODE == 'all_available' and all_files_complete and readiness['ready_for_expanded_training'])
split_manifest = {'version':i11d.VERSION,'run_mode':RUN_MODE,'can_train':CAN_TRAIN,
    'data_hash':readiness['all_data_hash'],'feature_names':i11t.FEATURES,
    'seed':SEED,'all_requested_files_complete':bool(all_files_complete),
    'real_fault_labels_available':False,'old_model_replaced':False}
i11d.write_json(I11_ROOT/'run_contract.json', split_manifest)
print('Expanded training enabled:', CAN_TRAIN)
if not CAN_TRAIN:
    print('Data-only result: inspect failed downloads/exclusions or switch from pilot to all_available and rerun.')

## 7. Causal features and controlled challenge labels

We use time gaps, missing-value counts, lag/rate changes, previous24-hour robust residuals,
elapsed flatline duration, previous3/6/24-hour slopes, seasonal time encodings, and buddy residuals.
No future data or raw labels enter features. Neighbours must be <=150km away, <=300m elevation difference,
independent (not <1km co-located aliases), and <=90 minutes old. Pressure buddies also require the same datum.
At least two usable buddies are required for a spatial residual; otherwise the feature is unavailable.
These are conservative engineering defaults, not universal meteorological standards.

Synthetic tests cover spikes, bias, drift, flatlines, noise, missing values and coherent regional perturbations.
Faults are injected **before feature building in target AND neighbour streams**; neighbours are not clean oracles.
Regional perturbations are synthetic weather challenges, not labelled real storms. Packet loss is distinct from
a missing sensor value: without a verified heartbeat contract, real archive gaps cannot establish communication faults.

In [ ]:
FEATURE_DIR = None
if CAN_TRAIN and RUN_TRAINING:
    FEATURE_DIR = i11t.build_features(I11_ROOT, seed=SEED)
    print('Checksummed per-station features:', FEATURE_DIR)
else:
    print('Feature/training stage skipped; data reports remain available.')

## 8. Fit and compare models

- Robust temporal QC comparator (non-learning score, separately calibrated).
- LightGBM without spatial features.
- LightGBM with causal spatial features.
- Original24-only **retrained** comparator, when enough old stations qualify. Same splits/features; not the old deployed binary.
- Optional CatBoost with spatial features, on T4 if available.

Per-station training mass is balanced and long episodes are downweighted. Training negatives are capped per station
for Colab memory; calibration and evaluation retain the full observed class prevalence. L1/L2 regularization and early stopping
control overfitting. New networks are not added without evidence they improve weak faults and weather false alarms.

Models/calibrators/policies are saved under this experiment only. A policy needs precision>=80% and <=0.05 false-positive
rows per observed station-day on the policy block. These are development targets, not official SIH score thresholds.
If none pass, the best descriptive F1 comparator is saved **with a failed gate**, never promoted.

In [ ]:
MODEL_OUTPUT = None
if FEATURE_DIR is not None:
    MODEL_OUTPUT = i11t.train_research(I11_ROOT, seed=SEED, use_catboost=USE_CATBOOST, gpu=GPU_AVAILABLE)
    policies = json.loads((MODEL_OUTPUT/'frozen_policies.json').read_text())
    display(pd.DataFrame(policies['policies'])[['model','threshold','precision','recall','f1','meets_development_budget']])
    gc.collect()

## 9. Frozen-policy confirmation (not a fresh blind test)

Reports point precision/recall/F1, average precision (PR-AUC), accuracy, Brier score, episode recall,
per-fault recall, per-station results and false alarms on synthetic weather challenges.
High accuracy alone is misleading because most rows are normal. Episode recall is any-point detection,
not onset accuracy. False-positive rows/day is not an incident-alert metric.

Do not retune against this report. Improvement is a measured comparison, not a promise.

In [ ]:
RESULT = None
if MODEL_OUTPUT is not None:
    RESULT = i11t.confirm_research(I11_ROOT, seed=SEED)
    display(pd.DataFrame(RESULT['comparison']))
    display(pd.read_csv(MODEL_OUTPUT/'iteration11_fault_recall.csv'))
else:
    print('No training results yet. Send the data-readiness report first.')

## 10. Optional repeatability checks

Extra seeds change synthetic challenge realizations and retrain models. They use the same station/time split.
Variation across seeds is not an independent real-world confidence interval. No automatic winning-seed selection.

In [ ]:
if RUN_EXTRA_SEEDS and CAN_TRAIN and RUN_TRAINING:
    repeated = []
    for extra_seed in [211, 311]:
        i11t.build_features(I11_ROOT, seed=extra_seed)
        i11t.train_research(I11_ROOT, seed=extra_seed, use_catboost=USE_CATBOOST, gpu=GPU_AVAILABLE)
        r = i11t.confirm_research(I11_ROOT, seed=extra_seed)
        repeated.extend([{'seed':extra_seed,**x} for x in r['comparison']])
        gc.collect()
    pd.DataFrame(repeated).to_csv(I11_ROOT/'repeat_seed_comparison.csv',index=False)
    display(pd.DataFrame(repeated))

## 11. Export reports to return for review

The report ZIP includes station eligibility/exclusions, acquisition status, spatial support,
split/feature contracts and results if training ran. Raw downloads and model binaries stay on Drive.
**Send this ZIP back.** Review must determine whether wider data improves station/fault recall without
excessive false alarms. No model is uploaded to the website by this notebook.

Before operational deployment: obtain compatible IMD AWS or owned-sensor T/P/RH streams, validate pressure/RH provenance,
collect independently verified fault/maintenance labels, test real weather extremes and packet SLAs, benchmark inference
resources, and explicitly promote a versioned model only after acceptance. Vercel website hosting alone does none of this.

In [ ]:
import zipfile
from google.colab import files
REPORT_ZIP = I11_ROOT/'SkyGuard_Iteration11_Data_Rebuild_Reports.zip'
with zipfile.ZipFile(REPORT_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ['discovery_receipt.json','station_catalog.csv','download_status.csv','station_quality.csv',
                 'neighbor_graph.csv','spatial_support.csv','data_readiness.json','run_contract.json','repeat_seed_comparison.csv']:
        p = I11_ROOT/name
        if p.exists(): z.write(p,name)
    for directory in sorted(I11_ROOT.glob('research_seed*')):
        for p in sorted(directory.iterdir()):
            if p.suffix in {'.csv','.json'}: z.write(p,p.relative_to(I11_ROOT))
print('Send back:',REPORT_ZIP)
files.download(str(REPORT_ZIP))